In [1]:
import chromadb
import pandas as pd
from langchain.docstore.document import Document
client = chromadb.PersistentClient(path="chroma-data")
collection = client.get_collection(name="eidc-metadata")
result = collection.get()
titles = [metadata["dataset_title"] for metadata in result["metadatas"]]

docs = [Document(page_content=result["documents"][i], metadata=result["metadatas"][i]) for i in range(len(result["documents"]))]
for doc in docs:
    doc.metadata["filename"] = doc.metadata["dataset_id"]
    doc.metadata["source"] = doc.metadata["dataset_id"]
docs = docs[:20]
len(docs)

20

In [2]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama
llm = ChatOllama(model='mistral-nemo', num_ctx=16384)
embeddings = OllamaEmbeddings(model='mistral-nemo', num_ctx=16384)

In [3]:
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from ragas.run_config import RunConfig
gen = TestsetGenerator.from_langchain(llm, llm, embeddings, run_config=RunConfig(max_workers=1, max_retries=1))
dist = {simple: 0.6, multi_context: 0.2, reasoning: 0.2}

In [4]:
import logging
import nest_asyncio
nest_asyncio.apply()
#logging.basicConfig(level=logging.INFO)
testset = gen.generate_with_langchain_docs(docs, 5, dist, is_async=False)

embedding nodes:   0%|          | 0/40 [00:00<?, ?it/s]

Filename and doc_id are the same for all nodes.


Generating:   0%|          | 0/5 [00:00<?, ?it/s]

max retries exceeded for SimpleEvolution(generator_llm=LangchainLLMWrapper(run_config=RunConfig(timeout=60, max_retries=15, max_wait=90, max_workers=16, thread_timeout=80.0, exception_types=(<class 'Exception'>,), log_tenacity=False)), docstore=InMemoryDocumentStore(splitter=<langchain_text_splitters.base.TokenTextSplitter object at 0x7bdba40f1d60>, nodes=[Node(metadata={'dataset_id': 'ad7babc3-6b43-4754-8981-edcf03769f11', 'dataset_title': 'Countryside Survey 1998 mapped estimates of Broad Habitat areas in Great Britain', 'eidc_metadata_key': 'description', 'filename': 'ad7babc3-6b43-4754-8981-edcf03769f11', 'source': 'ad7babc3-6b43-4754-8981-edcf03769f11'}, page_content='The dataset entitled "Countryside Survey 1998 mapped estimates of Broad Habitat areas in Great Britain" contains the following information in it\'s "description" metadata field: This dataset consists of stock (area) data for Broad Habitats across Great Britain in 1998 in a 1km grid format. The data are national estim

In [5]:
df = testset.to_pandas()
df.to_csv("eidc_rag_test_set.csv", index=False)

In [6]:
df

,question,contexts,ground_truth
0,What are the spatial and temporal coverage of ...,"[The dataset entitled ""Gridded estimates of da...",The spatial coverage of the gridded estimates ...
1,What are the 21 land cover classes in the 'Lan...,"[The dataset entitled ""Land Cover Map 2020 (10...",The answer to given question is not present in...
2,Where was the predictive model used to create ...,"[The dataset entitled ""Woody linear features f...",The predictive model was developed at the Cent...
3,What sensors & auto-checks in COSMOS-UK mitiga...,"[The dataset entitled ""Daily and sub-daily hyd...",The presence of snow leads to erroneously high...


In [29]:
df = pd.read_csv("ragas-testset.csv")
df

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,What was the average weed abundance across the...,"['The dataset entitled ""Abundance of weeds in ...",The answer to given question is not present in...,simple,[{'dataset_id': '6762f1b5-2bcc-4062-bff6-e560d...,True
1,How many harvests were conducted in total to m...,"['The dataset entitled ""Grass productivity dat...",Three different harvests were conducted in tot...,simple,[{'dataset_id': '6e395915-ab5c-43f4-b4de-c9a3c...,True
2,What specific parameters are recorded for each...,"['The dataset entitled ""UK Environmental Chang...",The specific parameters recorded for each tree...,simple,[{'dataset_id': '94aef007-634e-42db-bc52-9aae8...,True
3,What are the specific types of structures and ...,"['The dataset entitled ""Building, infrastructu...",The GIS shapefiles include information about b...,simple,[{'dataset_id': 'a763e254-c249-4934-b0fb-c3b80...,True
4,What are the estimated annual loads of nitroge...,"['The dataset entitled ""Non-agricultural pollu...",The answer to given question is not present in...,simple,[{'dataset_id': 'eb73ca31-7eb9-479c-96be-6063e...,True
...,...,...,...,...,...,...
95,What are the water quality parameters measured...,"['The dataset entitled ""Weekly water quality d...",The water quality parameters measured in the d...,simple,[{'dataset_id': 'cf10ea9a-a249-4074-ac0c-e0c30...,True
96,What are the four scenarios projected for land...,"['The dataset entitled "" Land use maps under t...",The four scenarios projected for land use in t...,simple,[{'dataset_id': 'a94640dc-fe21-4c38-936b-d62df...,True
97,What were the national estimates of Broad Habi...,"['The dataset entitled ""Countryside Survey 199...",The national estimates of Broad Habitat areas ...,simple,[{'dataset_id': '53ef00f4-e0c5-4095-850e-d4c47...,True
98,What does the 'Generate_data_for_island_model_...,"['The dataset entitled ""Code for generating da...",The 'Generate_data_for_island_model_weak_selec...,simple,[{'dataset_id': 'aec1e55f-f6ac-4673-9e93-706b4...,True


In [30]:
from rag.wrappers import RagPipelineWrapper
import yaml
with open("config.yml", "r") as config_file:
    config = yaml.safe_load(config_file)
pipeline_file = f"{config["pipelines-dir"]}/{config["rag-demo"]["pipeline"]}"
chroma_path = config["vector-db"]["path"]
collection = config["vector-db"]["collection"]
prompt = config["rag-demo"]["prompt"]
rag_pipe = RagPipelineWrapper(
    pipeline_file,
    chroma_path=chroma_path,
    collection=collection,
    prompt=prompt,
)

In [31]:
eval_output = []
for i, row in df.sample(30).iterrows():
    answer, documents = rag_pipe.query_get_contexts(row["question"])
    result = {
        "question": row["question"],
        "ground_truth": row["ground_truth"],
        "answer": answer,
        "contexts": [doc.content for doc in documents],
    }
    eval_output.append(result)
eval_df = pd.DataFrame(eval_output)

In [32]:
from datasets import Dataset
eval_dataset = Dataset.from_pandas(eval_df)

In [33]:
eval_dataset

Dataset({
    features: ['question', 'ground_truth', 'answer', 'contexts'],
    num_rows: 30
})

In [34]:
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_utilization,
    context_recall,
    context_entity_recall,
    answer_similarity,
    answer_correctness,
)
from ragas import evaluate
from ragas.run_config import RunConfig
import nest_asyncio

nest_asyncio.apply()
result = evaluate(
    eval_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_utilization,
        context_recall,
        context_entity_recall,
        answer_similarity,
        answer_correctness,
    ],
    llm=llm,
    embeddings=embeddings,
    is_async=False,
    raise_exceptions=False,
    run_config=RunConfig(max_workers=1),
)
result

Evaluating:   0%|          | 0/240 [00:00<?, ?it/s]

Failed to parse output. Returning None.


{'faithfulness': 0.6308, 'answer_relevancy': 0.3331, 'context_precision': 0.5328, 'context_utilization': 0.5302, 'context_recall': 0.7146, 'context_entity_recall': 0.2554, 'answer_similarity': 0.2784, 'answer_correctness': 0.2502}

In [35]:
eval_results = result.to_pandas()

In [42]:
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "gridon"
fig = go.Figure()
metrics = [metric for metric in eval_results.columns.to_list() if metric not in ["question", "ground_truth", "answer", "contexts"]]
for metric in metrics:
    fig.add_trace(go.Violin(y=eval_results[metric], name=metric, points="all", box_visible=True, meanline_visible=True))
fig.update_layout(title_text="RAG Evaluation Metrics (LLama 3.1)")
fig.update_yaxes(range=[-0.02,1.02])
fig.show()